# T05 — Multi-head attention

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** varias relaciones a la vez, sin coste extra  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Explicar por qué h cabezas de dimensión d/h cuestan lo mismo que una de dimensión d.
2. Observar que cabezas distintas producen distribuciones distintas.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

Una sola cabeza tiene que decidir una única forma de relacionar palabras. Varias cabezas permiten que una siga la concordancia, otra la dependencia sintáctica y otra la correferencia — y luego se concatena todo.


## 5. Concepto mínimo

```text
MultiHead(X) = Concat(head₁, …, head_h) · W^O
head_i = Attention(X·W_i^Q, X·W_i^K, X·W_i^V),   d_k = d_v = d_model / h
```

En el modelo base del paper: `d_model = 512`, `h = 8`, `d_k = 64`.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from ai_evolution.papers_lab import multi_head_attention

X = [[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0]]
r = multi_head_attention(X, heads=2)
for i, cabeza in enumerate(r['heads']):
    print(f'cabeza {i}:')
    for fila in cabeza:
        print('   ', [round(w, 3) for w in fila])

## 7. Predicción antes de ejecutar

Las dos cabezas ven mitades distintas del vector. ¿Producirán la misma distribución de atención?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
for h in (1, 2, 4):
    d_model = 8
    print(f'h={h} → d_k = {d_model // h} · parámetros de proyección ≈ {3 * d_model * d_model} (constante)')

## 9. Salida interpretable

El número de parámetros no depende de `h`: partir `d_model` en más cabezas no cuesta más memoria. Lo que cambia es la **capacidad de especialización**, no el presupuesto.


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
try:
    multi_head_attention([[1.0] * 5], heads=2)
except ValueError as exc:
    print('ValueError:', exc)
print('→ d_model debe ser divisible entre h; no es una convención, es aritmética')

## 12. Corrección


In [ ]:
r = multi_head_attention([[1.0, 0.5, 0.0, 0.2], [0.1, 0.9, 0.3, 0.4]], heads=2)
print('salida concatenada:', [[round(v, 3) for v in fila] for fila in r['output']])

## 13. Desafío guiado

Interpreta cada cabeza sobre una frase de 6 tokens y decide si alguna es prescindible (ablación).


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **varias relaciones a la vez, sin coste extra**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Falta un detalle grave: hasta aquí el modelo no sabe en qué ORDEN venían los tokens (T06).
